# Seq2Seq LSTM with PyTorch

An encoder-decoder LSTM maps one sequence to another. The encoder reads the input and the decoder generates the output starting only from the final encoder state, so that state has to hold the whole input. This notebook uses the task of reversing a sequence to show how far a single vector can go. See "LSTM with Attention" in the next lecture for the solution to this limitation.

**Aprendizado Profundo para Reconhecimento Visual** — Ifes Campus Serra
Autor: Thiago M. Paixão

In [ ]:
# If True, test is running on Colab. Otherwise, test is assumed to be offline.
TEST_ON_COLAB = True
FOLDERNAME = 'Colab Notebooks/Aprendizado Profundo para Reconhecimento Visual/07_rnn'  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount('/content/drive')
    os.chdir(f'/content/drive/My Drive/{FOLDERNAME}')

## Artificial dataset

The task is to reverse a sequence of digits. All sequences in a batch have the same length, so no padding is needed. The only special token is `sos_token`, which starts the decoder.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

vocab_size = 10  # digits from 0 to 9
sos_token = vocab_size  # start-of-sequence token, used as the first decoder input


def make_batch(batch_size, seq_len):
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    y = x.flip(dims=[1])
    return x, y


x, y = make_batch(1, 8)
print("Input: ", x[0].tolist())
print("Target:", y[0].tolist())

## Model

Training and generation use the decoder in different ways:

- **Training (teacher forcing):** the decoder input is the target shifted by one position. All inputs are known in advance, so `nn.LSTM` processes the whole sequence in a single call.
- **Generation:** each input is the previous prediction, which does not exist in advance. The same `nn.LSTM` is called one step at a time, and the state is passed from one call to the next.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, hidden_size)
        self.encoder = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, y):
        """Teacher forcing: returns the logits, of shape (N, T, C)."""
        _, state = self.encoder(self.embedding(x))  # state is all the decoder knows about x
        sos = torch.full_like(y[:, :1], sos_token)
        decoder_input = torch.cat([sos, y[:, :-1]], dim=1)  # (N, T): SOS, y_0, ..., y_{T-2}
        states, _ = self.decoder(self.embedding(decoder_input), state)
        return self.output(states)

    @torch.no_grad()
    def generate(self, x):
        """Greedy generation: returns the predicted tokens, of shape (N, T)."""
        _, state = self.encoder(self.embedding(x))
        token = torch.full_like(x[:, :1], sos_token)  # (N, 1): a sequence with a single step
        predictions = []
        for _ in range(x.size(1)):
            step, state = self.decoder(self.embedding(token), state)
            token = self.output(step).argmax(dim=2)
            predictions.append(token)
        return torch.cat(predictions, dim=1)

## Training

The same model is trained on three sequence lengths. The size of the state does not change, so longer sequences have to fit in the same vector.

In [ ]:
def train(seq_len, num_steps=1500, batch_size=128, learning_rate=1e-3):
    model = Seq2Seq().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_history = []
    for step in range(num_steps):
        x, y = make_batch(batch_size, seq_len)
        logits = model(x, y)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())
    return model, loss_history


lengths = [5, 10, 20]
models, histories = {}, {}
for seq_len in lengths:
    models[seq_len], histories[seq_len] = train(seq_len)
    print(f"Length {seq_len}: final loss {histories[seq_len][-1]:.4f}")

for seq_len in lengths:
    plt.plot(histories[seq_len], label=f"Length {seq_len}")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss History")
plt.show()

## Test

At test time the decoder only sees its own predictions. Token accuracy counts correct digits, and sequence accuracy counts sequences with all digits correct.

In [ ]:
accuracy_by_position = {}
for seq_len in lengths:
    x, y = make_batch(1000, seq_len)
    correct = models[seq_len].generate(x) == y
    accuracy_by_position[seq_len] = correct.float().mean(dim=0).cpu()
    print(f"Length {seq_len}: token accuracy={correct.float().mean():.3f}, "
          f"sequence accuracy={correct.all(dim=1).float().mean():.3f}")

x, y = make_batch(1, 20)
print("Target:   ", y[0].tolist())
print("Predicted:", models[20].generate(x)[0].tolist())

The plot shows the accuracy at each output position. A random guess is right 10% of the time. Compare where each curve falls: errors are not spread evenly along the sequence.

In [ ]:
for seq_len in lengths:
    plt.plot(range(1, seq_len + 1), accuracy_by_position[seq_len], marker="o", label=f"Length {seq_len}")
plt.axhline(1 / vocab_size, color="gray", linestyle="--", label="Random guess")
plt.xlabel("Output position")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy by output position")
plt.show()